# Treino do detector de agave — Colab

Use quando a GPU local não for confiável.

**Antes de subir**, no seu PC:

```
python tools/preparar_colab.py --tiles data/tiles
```

Isso verifica o dataset, corrige o `path:` do `data.yaml` (o caminho absoluto
do Windows não existe aqui) e gera `data/tiles.zip`.

**No Colab:** menu `Ambiente de execução` → `Alterar tipo de ambiente` → **GPU**.

## 1. Confirmar a GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import torch
assert torch.cuda.is_available(), (
    'Sem GPU. Ambiente de execucao -> Alterar tipo -> GPU, e rode de novo.')
p = torch.cuda.get_device_properties(0)
print(f'{p.name} | {p.total_memory/1024**3:.1f} GB')

# Diferente da GTX 1660, a T4 TEM Tensor Cores: aqui o AMP funciona e acelera.
AMP = p.major >= 7 and '16' not in p.name
print('mixed precision (amp):', AMP)

## 2. Instalar o Ultralytics

In [ ]:
%pip install -q ultralytics
import ultralytics; print('ultralytics', ultralytics.__version__)

## 3. Subir o dataset

Duas opções. A do Drive é melhor se você for treinar mais de uma vez —
o upload direto se perde quando a sessão cai.

In [ ]:
# --- OPCAO A: upload direto (simples, some quando a sessao cai) ---
from google.colab import files
enviados = files.upload()          # escolha data/tiles.zip
ZIP = next(iter(enviados))

# --- OPCAO B: Google Drive (recomendado) ---
# from google.colab import drive
# drive.mount('/content/drive')
# ZIP = '/content/drive/MyDrive/tiles.zip'

In [ ]:
import zipfile, pathlib

with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content/data')

DATA = '/content/data/tiles/data.yaml'
raiz = pathlib.Path('/content/data/tiles')
for split in ('train', 'val'):
    n_i = len(list((raiz/split/'images').glob('*.jpg')))
    n_cx = sum(len([l for l in f.read_text().splitlines() if l.strip()])
               for f in (raiz/split/'labels').glob('*.txt'))
    print(f'{split}: {n_i} tiles, {n_cx} caixas')

# o data.yaml precisa apontar para onde o zip foi extraido
texto = pathlib.Path(DATA).read_text()
texto = '\n'.join(f'path: {raiz}' if l.startswith('path:') else l
                  for l in texto.splitlines())
pathlib.Path(DATA).write_text(texto + '\n')
print('\n' + texto)

## 4. Treinar

`IMGSZ` deve ser o tamanho do tile, ou metade dele se o tile for grande.
Com tile 2048, treinar em 1024 mantém a planta com ~160px na rede e cabe
folgado na T4.

In [ ]:
MODELO = 'yolov8s.pt'   # yolov8m.pt se sobrar VRAM e tempo
IMGSZ  = 1024
BATCH  = 8              # T4 16GB: 8 em 1024. Se der OOM, caia para 4
EPOCAS = 150

from ultralytics import YOLO

modelo = YOLO(MODELO)
modelo.train(
    data=DATA, epochs=EPOCAS, imgsz=IMGSZ, batch=BATCH,
    device=0, amp=AMP, workers=2, name='agave_v1', exist_ok=True,
    patience=50, save_period=10,   # checkpoint a cada 10 epocas: a sessao pode cair
    # augmentacao para vista de cima: nao existe 'para cima' numa foto de drone
    degrees=180.0, fliplr=0.5, flipud=0.5, scale=0.5, translate=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,   # variacao de luz e o que mais quebra
    mosaic=1.0, close_mosaic=15,
)
print('\nsalvo em:', modelo.trainer.save_dir)

## 5. Ver o resultado

In [ ]:
from IPython.display import Image, display
import pathlib

saida = pathlib.Path(modelo.trainer.save_dir)
m = modelo.trainer.metrics
map50 = m.get('metrics/mAP50(B)', 0)
print(f"mAP50    {map50:.3f}")
print(f"mAP50-95 {m.get('metrics/mAP50-95(B)', 0):.3f}")
print(f"precisao {m.get('metrics/precision(B)', 0):.3f}")
print(f"recall   {m.get('metrics/recall(B)', 0):.3f}")

if map50 >= 0.70:
    print('\n-> Bom. Ja pre-anota bem: siga para o predict_tiled.')
elif map50 >= 0.50:
    print('\n-> Razoavel para 9 imagens. Serve para pre-anotar; corrija e retreine.')
else:
    print('\n-> Baixo. Olhe val_batch0_pred.jpg abaixo antes de tirar conclusao.')

for nome in ('results.png', 'confusion_matrix_normalized.png',
             'val_batch0_pred.jpg', 'val_batch0_labels.jpg'):
    f = saida / nome
    if f.exists():
        print('\n' + nome)
        display(Image(filename=str(f), width=900))

## 6. Baixar os pesos

**Faça isso antes de fechar.** A sessão do Colab é apagada e leva o treino junto.

In [ ]:
from google.colab import files
best = saida / 'weights' / 'best.pt'
print(best, f'({best.stat().st_size/1024**2:.1f} MB)')
files.download(str(best))

# Se montou o Drive, melhor copiar para la — download do navegador as vezes falha:
# import shutil; shutil.copy(best, '/content/drive/MyDrive/agave_best.pt')

## De volta no seu PC

Coloque o `best.pt` em `runs/detect/agave_v1/weights/` e siga o loop normal.
A inferência roda bem em CPU, então a GPU com defeito não te trava aqui:

```
python src/predict_tiled.py --weights best.pt --images data/raw/nao_rotuladas \
    --out data/preanot --tile 2048 --imgsz 1024 --conf 0.20 \
    --save-labels --classe 1 --device cpu
```

E meça contra as 9 imagens que você já rotulou, para saber se o modelo presta:

```
python src/predict_tiled.py --weights best.pt --images data/raw/images \
    --out out/yolo --tile 2048 --imgsz 1024 --conf 0.25 --save-labels --device cpu
python src/eval_proposals.py --pred out/yolo/labels --gt data/raw/labels \
    --images data/raw/images --out out/erros
```